In [ ]:
import numpy as np
import pandas as pd
import re
import string

In [ ]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [ ]:
#open the downloaded english stop words file
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines() #take line by line as a list

In [ ]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [ ]:
#create a function includeing all steps in preprocessing
def preprocessing(text):
    data = pd.DataFrame([text], columns=["tweet"])
    # 1. convert uppercase to lowercase in tweet column using lambda function (bcz only tweet has letters)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    #2
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x,flags=re.MULTILINE) for x in x.split()))
    #3. remove punctuation by creating a function (we can use lambda function too)
    data["tweet"] = data["tweet"].apply(remove_punctuations)
    #4. remove numbers
    data["tweet"] = data["tweet"].str.replace(r'\d+', '', regex=True)
    #remove stopwords
    data["tweet"] = data["tweet"].apply(lambda x: " ". join(x for x in x.split() if x not in sw))
    #stem
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]

In [ ]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
tokens = vocab[0].tolist()

In [ ]:
def vectorizer(ds, vocabulary):
    # Create an empty list to store the vectorized sentences
    vectorized_list = []

    # Loop through each sentence in the dataset
    for sentence in ds:
        # Create a vector of zeros with the same length as the vocabulary
        # Each position represents a word in the vocabulary
        sentence_list = np.zeros(len(vocabulary))

        # Loop through each word in the vocabulary
        for i in range(len(vocabulary)):

            # Check if the vocabulary word exists in the sentence
            # sentence.split() converts the sentence into a list of words
            if vocabulary[i] in sentence.split():

                # If the word is present, set the corresponding position to 1
                # This represents the presence of that word in the sentence
                sentence_list[i] = 1

        # After checking all vocabulary words, add the vector to the list
        vectorized_list.append(sentence_list)

    # Convert the list of vectors into a NumPy array with float32 datatype
    vectorized_list_new = np.asarray(vectorized_list, dtype=np.float32)

    # Return the vectorized dataset
    return vectorized_list_new

In [ ]:
import pickle

In [ ]:
#open the model
with open('../static/model/model2.pickle', 'rb') as f:
    model = pickle.load(f)

In [ ]:
#take as positive or negative
def get_prediction(vectorized_text):
    prediction = model.predict(vectorized_text)
    if prediction == 1:
        return 'negative'
    else:
        return 'positive'